In [ ]:
import os
import pandas as pd

In [ ]:
#step 1 and 2 genomic files here
data='/home/jupyter/workspaces/duplicateofinfectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results'

#pheno and covar file here
results='/home/jupyter/workspaces/duplicateofinfectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results'

In [ ]:
!lsof tmp_acaf_step1_split2.vcf.gz

In [ ]:
!ps aux | grep bcftools


In [ ]:
!zcat tmp_acaf_step1_split2.vcf.gz | tail -n5

In [ ]:
!df -h

In [ ]:
#setup regenie

In [ ]:
!conda config --prepend pkgs_dirs $HOME/.conda/pkgs
!conda config --prepend envs_dirs $HOME/.conda/envs

!conda config --show pkgs_dirs
!conda config --show envs_dirs

In [ ]:
%%bash
# If you already have conda, just do the 'create' line; otherwise see the conda note below.
conda create -n regenie4 -y -c conda-forge -c bioconda regenie

regenie --version  
# should print 4.1.x
which -a regenie           # ensure the conda one is first on PATH


In [ ]:
!conda run -n regenie4 regenie --version
!conda run -n regenie4 which -a regenie

In [ ]:
#Fix genomic files

In [ ]:
#fixstep 1 psam file

def fix_psam_for_regenie(psam_path):
    
   
    # Read whitespace-delimited .psam, keep everything as string
    df = pd.read_csv(psam_path, delim_whitespace=True, dtype=str)

    # Case 1: already has '#FID' and 'IID' (your current case)
    if "#FID" in df.columns and "IID" in df.columns:
        pass

    # Case 2: older file where you only had '#IID'
    elif "#IID" in df.columns and "#FID" not in df.columns:
        # create FID = IID and rename '#IID' -> 'IID'
        df.insert(0, "#FID", df["#IID"])
        df.rename(columns={"#IID": "IID"}, inplace=True)

    else:
        raise ValueError(f"PSAM header columns unexpected: {df.columns.tolist()}")

    # Ensure SEX column exists
    if "SEX" not in df.columns:
        df["SEX"] = "0"  # unknown sex
    else:
        # Fill NaNs and empty strings with "0"
        df["SEX"] = df["SEX"].fillna("0")
        df["SEX"] = df["SEX"].replace("", "0")

    # Write back as tab-delimited, keeping '#FID' as the first column name
    df.to_csv(psam_path, sep="\t", index=False)

    print("Fixed PSAM written to:", psam_path)
    print(df.head())

fix_psam_for_regenie("/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_threshold.step1_snps.psam")

In [ ]:
%%bash

PFX="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_threshold.step1_snps"

plink2 \
  --pfile "${PFX}" \
  --export vcf-4.2 bgz \
  --out tmp_acaf_step1

In [ ]:
!bcftools norm -m-any -@ 16 tmp_acaf_step1.vcf.gz -Oz -o tmp_acaf_step1_split.vcf.gz
!bcftools index -@ 16 tmp_acaf_step1_split.vcf.gz

In [ ]:
%%bash

plink2 \
  --vcf tmp_acaf_step1_split.vcf.gz \
  --threads 16 \
  --make-bed \
  --out acaf_step1_regenie

In [ ]:
%%bash

plink2 \
  --vcf tmp_acaf_step1_split2.vcf.gz \
  --threads 16 \
  --make-bed \
  --out acaf_step1_regenie

In [ ]:
#run regenie

In [ ]:
%%writefile regenie_test.sh

#!/usr/bin/env bash
set -euo pipefail

# Use REGENIE from your conda env (no need to activate)
export PATH="$HOME/.conda/envs/regenie4/bin:$PATH"


# Show which regenie we will use (sanity check)
echo "Using regenie at: $(which regenie)"


# Local paths

PLINK1="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_threshold.step1_snps_bed"     # prefix only (has .bed/.bim/.fam)

PLINK2="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/acaf_v7_plink_files/acaf_threshold.chr16.bed"       # prefix only (has .bed/.bim/.fam)


MERGED_TSV='/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/lupus_matched_ALL.regenie.txt'        # columns: FID IID meningitis sex age
OUT_DIR='/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/regenie_results'

mkdir -p "$OUT_DIR"


#STEP 1 - building moc

regenie \
  --step 1 \
  --bed "${PLINK1}" \
  --bt --loocv \
  --phenoFile "${MERGED_TSV}" \
  --phenoCol case \
  --covarFile "${MERGED_TSV}" \
  --covarColList sex,age,age2,age_sex,age2_sex,PC{1:16} \
  --lowmem \
  --bsize 1000 \
  --threads 16 \
  --gz \
  --print-pheno \
  --out "${OUT_DIR}/lupus_all_fit_step1"

# STEP 2 — single-variant test
regenie \
  --step 2 \
  --bed "${PLINK2}" \
  --bt --loocv \
  --phenoFile "${MERGED_TSV}" 
  --phenoCol case \
  --covarFile  "${MERGED_TSV}" \
  --covarColList sex,age,age2,age_sex,age2_sex,PC{1:16} \
  --pred "${OUT_DIR}/lupus_fit_step1_pred.list" \
  --bsize 500 \
  --threads 16 \
  --out "${OUT_DIR}/lupus_all_assoc_"



In [ ]:
%%bash

chmod +x regenie_test.sh
./regenie_test.sh



In [ ]:
#view files
!head -n 100 /home/jupyter/workspaces/duplicateofinfectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/regenie_results/hepb_all_assoc_acaf_threshold.chr10_case.regenie
